# Stock Prediction Analysis

This notebook mirrors the Streamlit app flow: data checks, feature engineering, training, metrics, and forecast output.
The process saves .pkl files in the models directory, the outpus were left for ilustrative purposes in case streamlit doesn't work.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.express as px

data_dir = Path('..') / 'data'
price = pd.read_csv(data_dir / 'price.csv')
news = pd.read_csv(data_dir / 'news.csv')

price['date'] = pd.to_datetime(price['date'], errors='coerce')
news['datetime'] = pd.to_datetime(news['datetime'], errors='coerce')
news['date'] = news['datetime'].dt.date
news['date'] = pd.to_datetime(news['date'], errors='coerce')

price.head()

,date,ticker,open,high,low,close,volume
0,2023-11-13,AAPL,185.820007,186.029999,184.210007,184.800003,43627500
1,2023-11-14,AAPL,187.699997,188.110001,186.300003,187.440002,60108400
2,2023-11-15,AAPL,187.850006,189.500000,187.779999,188.009995,53790500
3,2023-11-16,AAPL,189.570007,190.960007,188.649994,189.710007,54412900
4,2023-11-17,AAPL,190.250000,190.380005,188.570007,189.690002,50922700


In [2]:
# Data overview
print('Price shape:', price.shape)
print('News shape:', news.shape)

print('Price date range:', price['date'].min(), '->', price['date'].max())
print('News datetime range:', news['datetime'].min(), '->', news['datetime'].max())

print('Price tickers:', price['ticker'].nunique())
print('News tickers:', news['ticker'].nunique())

print('Price missing values:\n', price.isna().sum())
print('News missing values:\n', news.isna().sum())

print('Price duplicate rows:', price.duplicated().sum())
print('News duplicate rows:', news.duplicated().sum())

price.groupby('ticker').size().sort_values(ascending=False)

Price shape: (1687, 7)
News shape: (4440, 5)
Price date range: 2023-11-13 00:00:00 -> 2024-10-28 00:00:00
News datetime range: 2023-11-13 09:12:41 -> 2024-10-29 19:34:15
Price tickers: 7
News tickers: 7
Price missing values:
 date      0
ticker    0
open      0
high      0
low       0
close     0
volume    0
dtype: int64
News missing values:
 datetime    0
ticker      0
headline    0
summary     1
date        0
dtype: int64
Price duplicate rows: 0
News duplicate rows: 1


ticker
AAPL     241
AMZN     241
GOOGL    241
META     241
MSFT     241
NVDA     241
TSLA     241
dtype: int64

## Price-News Alignment
Check how often each ticker/date has news coverage.

In [3]:
news_daily = news.groupby(['ticker', 'date']).size().reset_index(name='news_count')
price_dates = price[['ticker', 'date']].drop_duplicates()

merged_counts = price_dates.merge(news_daily, on=['ticker', 'date'], how='left')
merged_counts['news_count'] = merged_counts['news_count'].fillna(0)

merged_counts['news_count'].describe()

# Share of days with at least one news item
coverage = (merged_counts['news_count'] > 0).mean()
coverage

np.float64(0.7362181387077653)

## News EDA
Inspect news coverage by ticker and time, plus text length diagnostics.

In [4]:
news['headline_len'] = news['headline'].fillna('').str.len()
news['summary_len'] = news['summary'].fillna('').str.len()

news_counts = news.groupby('ticker').size().sort_values(ascending=False)
news_counts

# News per day
news_per_day = news.groupby('date').size().reset_index(name='count')
fig = px.line(news_per_day, x='date', y='count', title='News Count Over Time')
fig.show()

# Text length distribution
fig = px.histogram(news, x='headline_len', nbins=50, title='Headline Length Distribution')
fig.show()
fig = px.histogram(news, x='summary_len', nbins=50, title='Summary Length Distribution')
fig.show()

## Price EDA
Explore returns, volume, and price trends by ticker.

In [5]:
price_sorted = price.sort_values(['ticker', 'date']).copy()
price_sorted['return'] = price_sorted.groupby('ticker')['close'].pct_change()

# Price trend by ticker
fig = px.line(price_sorted, x='date', y='close', color='ticker', title='Close Price Over Time')
fig.show()

# Return distribution
fig = px.histogram(price_sorted.dropna(subset=['return']), x='return', color='ticker', nbins=60, title='Daily Return Distribution')
fig.show()

# Volume over time
fig = px.line(price_sorted, x='date', y='volume', color='ticker', title='Volume Over Time')
fig.show()

## Full Pipeline (Notebook Fallback)
Run the end-to-end pipeline here if the Streamlit app is unavailable.

In [6]:
def _check_import(name, install_hint):
    try:
        __import__(name)
        return True, None
    except Exception as exc:
        return False, f"{name} import failed: {exc}. Install: {install_hint}"

checks = [
    ("xgboost", "conda install -c conda-forge libomp xgboost"),
    ("torch", "pip install torch"),
    ("torchvision", "pip install torchvision"),
    ("nbformat", "pip install 'nbformat>=4.2.0'"),
]

results = []
for name, hint in checks:
    ok, msg = _check_import(name, hint)
    results.append((name, ok, msg))

results

[('xgboost', True, None),
 ('torch', True, None),
 ('torchvision', True, None),
 ('nbformat', True, None)]

## Troubleshooting
Quick dependency checks to prevent common runtime errors.

In [7]:
import sys
from pathlib import Path

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.baseline_models import BaselineModels
from src.config import PRICE_FILE, DATA_DIR, NEWS_CACHE_FILE
from src.data_loader import DataLoader
from src.data_validator import DataValidator
from src.evaluator import Evaluator
from src.feature_engineering import FeatureEngineer
from src.facade import StockPredictionFacade
from src.model_trainer import ModelTrainer
from src.news_processor import NewsProcessor

facade = StockPredictionFacade(
    data_loader=DataLoader(PRICE_FILE, DATA_DIR / 'news.csv'),
    data_validator=DataValidator(),
    news_processor=NewsProcessor(
        embedding_model_name='all-MiniLM-L6-v2',
        sentiment_model_name='ProsusAI/finbert',
        cache_path=NEWS_CACHE_FILE,
    ),
    feature_engineer=FeatureEngineer(),
    model_trainer=ModelTrainer(),
    evaluator=Evaluator(),
    baseline_models=BaselineModels(),
)

# Run full pipeline (may take time on first run)
facade.run_full_pipeline()

facade.metrics

/opt/anaconda3/envs/sp1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'classification': {'accuracy': 0.49206349206349204,
  'precision': 0.5771812080536913,
  'recall': 0.5695364238410596,
  'f1': 0.5733333333333334,
  'f1_macro': 0.47294117647058825,
  'balanced_accuracy': 0.47288702380171793,
  'confusion_matrix': [[38, 63], [65, 86]],
  'classification_report': {'0': {'precision': 0.36893203883495146,
    'recall': 0.37623762376237624,
    'f1-score': 0.37254901960784315,
    'support': 101.0},
   '1': {'precision': 0.5771812080536913,
    'recall': 0.5695364238410596,
    'f1-score': 0.5733333333333334,
    'support': 151.0},
   'accuracy': 0.49206349206349204,
   'macro avg': {'precision': 0.4730566234443214,
    'recall': 0.47288702380171793,
    'f1-score': 0.47294117647058825,
    'support': 252.0},
   'weighted avg': {'precision': 0.49371626324776785,
    'recall': 0.49206349206349204,
    'f1-score': 0.4928602552131964,
    'support': 252.0}},
  'roc_auc': 0.47249360697659176},
 'regression': {'mae': 0.01451443730094983,
  'rmse': 0.0237027992

## Streamlit Parity (Notebook)
This section mirrors the Streamlit app flow, including metrics, per-ticker breakdown, and forecast chart.

In [8]:
# Rebuild the facade the same way app.py does
from src.baseline_models import BaselineModels
from src.config import DATA_DIR, NEWS_CACHE_FILE, PRICE_FILE
from src.data_loader import DataLoader
from src.data_validator import DataValidator
from src.evaluator import Evaluator
from src.feature_engineering import FeatureEngineer
from src.facade import StockPredictionFacade
from src.model_trainer import ModelTrainer
from src.news_processor import NewsProcessor

facade = StockPredictionFacade(
    data_loader=DataLoader(PRICE_FILE, DATA_DIR / 'news.csv'),
    data_validator=DataValidator(),
    news_processor=NewsProcessor(
        embedding_model_name='all-MiniLM-L6-v2',
        sentiment_model_name='ProsusAI/finbert',
        cache_path=NEWS_CACHE_FILE,
    ),
    feature_engineer=FeatureEngineer(),
    model_trainer=ModelTrainer(),
    evaluator=Evaluator(),
    baseline_models=BaselineModels(),
)

# Step 1: load and validate
facade.load_data()
validation_report = facade.validate_data()
validation_report

{'price_missing_values': {'date': 0,
  'ticker': 0,
  'open': 0,
  'high': 0,
  'low': 0,
  'close': 0,
  'volume': 0},
 'news_missing_values': {'datetime': 0,
  'ticker': 0,
  'headline': 0,
  'summary': 1,
  'date': 0},
 'price_duplicate_rows': 0,
 'news_duplicate_rows': 1,
 'price_invalid_dates': 0,
 'news_invalid_datetimes': 0,
 'news_row_count': 4440,
 'news_group_total_rows': 4440,
 'news_group_count_mismatch': 0,
 'news_group_count_summary': {'count': 1387.0,
  'mean': 3.2011535688536408,
  'std': 3.0941081367375114,
  'min': 1.0,
  '25%': 1.0,
  '50%': 2.0,
  '75%': 4.0,
  'max': 32.0},
 'news_group_count_sample': {('TSLA', Timestamp('2024-06-13 00:00:00')): 32,
  ('NVDA', Timestamp('2024-03-18 00:00:00')): 28,
  ('AAPL', Timestamp('2024-06-10 00:00:00')): 27,
  ('NVDA', Timestamp('2024-02-22 00:00:00')): 21,
  ('NVDA', Timestamp('2024-06-25 00:00:00')): 19},
 'tickers_with_few_rows': {},
 'news_tickers_without_price': []}

In [9]:
# Step 2: feature prep
facade.prepare_features()
facade.processed_df.head()

,date,ticker,open,high,low,close,volume,return,return_lag_1,return_lag_2,...,latest_emb_374,latest_emb_375,latest_emb_376,latest_emb_377,latest_emb_378,latest_emb_379,latest_emb_380,latest_emb_381,latest_emb_382,latest_emb_383
0,2023-11-13,AAPL,185.820007,186.029999,184.210007,184.800003,43627500,NaN,NaN,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2023-11-14,AAPL,187.699997,188.110001,186.300003,187.440002,60108400,0.014286,NaN,NaN,...,0.039579,-0.019216,0.009826,-0.019879,-0.043995,0.004803,-0.003361,-0.038330,0.016279,-0.007533
2,2023-11-15,AAPL,187.850006,189.500000,187.779999,188.009995,53790500,0.003041,0.014286,NaN,...,-0.006056,-0.077467,-0.003118,-0.062171,-0.032421,0.042881,0.004560,-0.143960,0.099428,0.042343
3,2023-11-16,AAPL,189.570007,190.960007,188.649994,189.710007,54412900,0.009042,0.003041,0.014286,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2023-11-17,AAPL,190.250000,190.380005,188.570007,189.690002,50922700,-0.000105,0.009042,0.003041,...,-0.027719,-0.018489,0.002613,-0.054123,-0.024011,0.008716,0.062154,-0.098993,0.087891,0.068253


In [10]:
# Weighted embedding example (formula + sample rows)
example = facade.processed_df[["ticker", "date", "news_count", "weighted_embedding", "embedding"]].head(3)
example

,ticker,date,news_count,weighted_embedding,embedding
0,AAPL,2023-11-13,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,AAPL,2023-11-14,1,"[0.027108490467071533, -0.010115629993379116, ...","[0.027108490467071533, -0.010115629993379116, ..."
2,AAPL,2023-11-15,1,"[-0.058220274746418, -0.06032750755548477, -0....","[-0.058220274746418, -0.06032750755548477, -0...."


In [11]:
columns_guide = [
    ("return", "Daily close-to-close return per ticker"),
    ("return_lag_1", "Return from the previous trading day"),
    ("return_lag_2", "Return from two trading days ago"),
    ("return_lag_3", "Return from three trading days ago"),
    ("ma_5", "5-day moving average of close"),
    ("ma_10", "10-day moving average of close"),
    ("ma_20", "20-day moving average of close"),
    ("volatility_5", "5-day rolling std of returns"),
    ("volatility_10", "10-day rolling std of returns"),
    ("volume_change", "Daily percent change in volume"),
    ("volume_ma_5", "5-day moving average of volume"),
    ("high_low_range", "High minus low"),
    ("close_open_body", "Close minus open"),
    ("body_ratio", "Body divided by high-low range"),
    ("close_to_high", "Close relative to high"),
    ("close_to_low", "Close relative to low"),
    ("day_of_week", "Day of week (0=Mon)"),
    ("news_count", "Number of news items per ticker/day"),
    ("positive_count", "Count of positive news items"),
    ("negative_count", "Count of negative news items"),
    ("neutral_count", "Count of neutral news items"),
    ("mean_sentiment_score", "Average sentiment confidence score"),
    ("max_sentiment_score", "Max sentiment confidence score"),
    ("mean_sentiment_numeric", "Mean sentiment label mapped to -1/0/1"),
    ("abs_mean_sentiment", "Average absolute sentiment label"),
    ("latest_sentiment_numeric", "Sentiment label of latest news item"),
    ("weighted_embedding", "Vector: weighted mean embedding for the day"),
    ("embedding", "Vector: embedding from latest news item"),
    ("weighted_emb_*", "Weighted embedding components (model-ready)"),
    ("latest_emb_*", "Latest embedding components (model-ready)"),
    ("target_next_return", "Next-day return (regression target)"),
    ("target_direction", "Next-day direction (classification target)"),
]

pd.DataFrame(columns_guide, columns=["column", "meaning"]).head(15)

,column,meaning
0,return,Daily close-to-close return per ticker
1,return_lag_1,Return from the previous trading day
2,return_lag_2,Return from two trading days ago
3,return_lag_3,Return from three trading days ago
4,ma_5,5-day moving average of close
5,ma_10,10-day moving average of close
6,ma_20,20-day moving average of close
7,volatility_5,5-day rolling std of returns
8,volatility_10,10-day rolling std of returns
9,volume_change,Daily percent change in volume


## Processed Columns Guide
Key engineered columns and an example of the weighted embedding calculation.

In [12]:
# Step 3: split, train, evaluate, and save
split = facade.split_data()
(
    X_train,
    X_val,
    X_test,
    y_train_cls,
    y_val_cls,
    y_test_cls,
    y_train_reg,
    y_val_reg,
    y_test_reg,
 ) = split

facade.train_models(X_train, y_train_cls, y_train_reg, X_val, y_val_cls, y_val_reg)
facade.evaluate_models(X_test, y_test_cls, y_test_reg)
facade.save_artifacts()

{'metrics': facade.metrics, 'per_ticker_count': len(facade.metrics_by_ticker)}

{'metrics': {'classification': {'accuracy': 0.49206349206349204,
   'precision': 0.5771812080536913,
   'recall': 0.5695364238410596,
   'f1': 0.5733333333333334,
   'f1_macro': 0.47294117647058825,
   'balanced_accuracy': 0.47288702380171793,
   'confusion_matrix': [[38, 63], [65, 86]],
   'classification_report': {'0': {'precision': 0.36893203883495146,
     'recall': 0.37623762376237624,
     'f1-score': 0.37254901960784315,
     'support': 101.0},
    '1': {'precision': 0.5771812080536913,
     'recall': 0.5695364238410596,
     'f1-score': 0.5733333333333334,
     'support': 151.0},
    'accuracy': 0.49206349206349204,
    'macro avg': {'precision': 0.4730566234443214,
     'recall': 0.47288702380171793,
     'f1-score': 0.47294117647058825,
     'support': 252.0},
    'weighted avg': {'precision': 0.49371626324776785,
     'recall': 0.49206349206349204,
     'f1-score': 0.4928602552131964,
     'support': 252.0}},
   'roc_auc': 0.47249360697659176},
  'regression': {'mae': 0.0145

In [13]:
# Per-ticker classification metrics table
per_ticker_rows = []
for ticker, metrics in facade.metrics_by_ticker.items():
    per_ticker_rows.append({
        "ticker": ticker,
        "accuracy": metrics.get("accuracy"),
        "precision": metrics.get("precision"),
        "recall": metrics.get("recall"),
        "f1": metrics.get("f1"),
        "f1_macro": metrics.get("f1_macro"),
        "roc_auc": metrics.get("roc_auc"),
        "balanced_accuracy": metrics.get("balanced_accuracy"),
        "support": metrics.get("support"),
        "support_pos": metrics.get("support_pos"),
        "support_neg": metrics.get("support_neg"),
    })
per_ticker_df = pd.DataFrame(per_ticker_rows)
per_ticker_df.sort_values("f1_macro", ascending=False).head(10)

,ticker,accuracy,precision,recall,f1,f1_macro,roc_auc,balanced_accuracy,support,support_pos,support_neg
1,AMZN,0.527778,0.647059,0.500000,0.564103,0.524476,0.555195,0.535714,36,22,14
4,MSFT,0.500000,0.550000,0.550000,0.550000,0.493750,0.531250,0.493750,36,20,16
2,GOOGL,0.527778,0.640000,0.666667,0.653061,0.456965,0.482639,0.458333,36,24,12
0,AAPL,0.472222,0.545455,0.571429,0.558140,0.451484,0.495238,0.452381,36,21,15
5,NVDA,0.500000,0.600000,0.652174,0.625000,0.437500,0.461538,0.441472,36,23,13
6,TSLA,0.500000,0.517241,0.789474,0.625000,0.437500,0.458204,0.482972,36,19,17
3,META,0.416667,0.545455,0.272727,0.363636,0.412587,0.344156,0.457792,36,22,14


In [14]:
# Step 4: prediction example
ticker = facade.processed_df['ticker'].iloc[0]
facade.predict_latest(ticker)

{'latest_close': 231.41000366210935,
 'predicted_direction': 0,
 'probability_up': 0.4361014664173126,
 'predicted_return': -0.0039917537942528725,
 'predicted_close': 230.48627190196305}

In [15]:
# Forecast chart (+1 day)
import plotly.graph_objects as go

ticker = facade.processed_df["ticker"].iloc[0]
hist_df = facade.processed_df[facade.processed_df["ticker"] == ticker].copy()
hist_df = hist_df.sort_values("date")

forecast = facade.predict_latest(ticker)
last_date = pd.to_datetime(hist_df["date"].iloc[-1])
next_date = last_date + pd.Timedelta(days=1)
last_close = float(hist_df["close"].iloc[-1])
predicted_close = float(forecast["predicted_close"])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=hist_df["date"],
        y=hist_df["close"],
        mode="lines",
        name="Historical Close",
        line=dict(color="#2b6cb0", width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=[last_date, next_date],
        y=[last_close, predicted_close],
        mode="lines+markers",
        name="Forecast (+1 day)",
        line=dict(color="#e53e3e", width=3, dash="dash"),
        marker=dict(size=8),
    )
)
fig.update_layout(title="Grafico de Close Prices", xaxis_title="Date", yaxis_title="Close")
fig

## Upload New Data (Notebook)
Provide new CSV paths to rebuild embeddings, features, and models from scratch.

In [16]:
# Replace these paths with your new files
new_price_path = Path("../data/price.csv")
new_news_path = Path("../data/news.csv")

new_price_df = pd.read_csv(new_price_path)
new_news_df = pd.read_csv(new_news_path)

new_price_df["date"] = pd.to_datetime(new_price_df["date"], errors="coerce")
new_news_df["datetime"] = pd.to_datetime(new_news_df["datetime"], errors="coerce")
new_news_df["date"] = new_news_df["datetime"].dt.date
new_news_df["date"] = pd.to_datetime(new_news_df["date"], errors="coerce")

# Recompute embeddings and retrain from scratch
facade.run_full_pipeline_on_data(new_price_df, new_news_df)
facade.metrics

Batches: 100%|██████████| 139/139 [00:04<00:00, 30.30it/s]


{'classification': {'accuracy': 0.49206349206349204,
  'precision': 0.5771812080536913,
  'recall': 0.5695364238410596,
  'f1': 0.5733333333333334,
  'f1_macro': 0.47294117647058825,
  'balanced_accuracy': 0.47288702380171793,
  'confusion_matrix': [[38, 63], [65, 86]],
  'classification_report': {'0': {'precision': 0.36893203883495146,
    'recall': 0.37623762376237624,
    'f1-score': 0.37254901960784315,
    'support': 101.0},
   '1': {'precision': 0.5771812080536913,
    'recall': 0.5695364238410596,
    'f1-score': 0.5733333333333334,
    'support': 151.0},
   'accuracy': 0.49206349206349204,
   'macro avg': {'precision': 0.4730566234443214,
    'recall': 0.47288702380171793,
    'f1-score': 0.47294117647058825,
    'support': 252.0},
   'weighted avg': {'precision': 0.49371626324776785,
    'recall': 0.49206349206349204,
    'f1-score': 0.4928602552131964,
    'support': 252.0}},
  'roc_auc': 0.47249360697659176},
 'regression': {'mae': 0.014509781958169063,
  'rmse': 0.023723446

## Embeddings Setup (Online or Offline)
Use one of the following cells depending on your connectivity.

In [17]:
# Option A: Online (downloads models and builds embeddings cache)
from src.config import DATA_DIR, NEWS_CACHE_FILE
from src.data_loader import DataLoader
from src.news_processor import NewsProcessor

news_df = DataLoader(DATA_DIR / 'price.csv', DATA_DIR / 'news.csv').load_news()
processor = NewsProcessor(
    embedding_model_name='all-MiniLM-L6-v2',
    sentiment_model_name='ProsusAI/finbert',
    cache_path=NEWS_CACHE_FILE,
    local_files_only=False,
    disable_hf_xet=True,
)
processed_news = processor.process_news(news_df, force_recompute=True)
processed_news[['ticker', 'date', 'sentiment_label']].head()

Batches: 100%|██████████| 139/139 [00:03<00:00, 37.09it/s]


,ticker,date,sentiment_label
0,AAPL,2023-11-13,negative
1,AAPL,2023-11-14,positive
2,AAPL,2023-11-16,neutral
3,AAPL,2023-11-16,neutral
4,AAPL,2023-11-16,positive


In [18]:
# Option B: Offline (requires existing cache file)
from src.config import DATA_DIR, NEWS_CACHE_FILE
from src.data_loader import DataLoader
from src.news_processor import NewsProcessor

news_df = DataLoader(DATA_DIR / 'price.csv', DATA_DIR / 'news.csv').load_news()
processor = NewsProcessor(
    embedding_model_name='all-MiniLM-L6-v2',
    sentiment_model_name='ProsusAI/finbert',
    cache_path=NEWS_CACHE_FILE,
    local_files_only=True,
    disable_hf_xet=True,
)
processed_news = processor.process_news(news_df, force_recompute=False)
processed_news[['ticker', 'date', 'sentiment_label']].head()

,ticker,date,sentiment_label
0,AAPL,2023-11-13,negative
1,AAPL,2023-11-14,positive
2,AAPL,2023-11-16,neutral
3,AAPL,2023-11-16,neutral
4,AAPL,2023-11-16,positive
